In [1]:
import os

In [2]:
%pwd

'c:\\MLProject\\TextSummarizer\\research'

In [3]:
os.chdir('../')

In [4]:
%pwd

'c:\\MLProject\\TextSummarizer'

In [1]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir : Path

In [2]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [3]:
class ConfiguarationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAM_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_roots])

    def get_data_ingestion_config(self)->DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_url=config.source_url,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config


**Components**

In [4]:
import os
import urllib.request as request
from textSummarizer.utils.common import get_size
import zipfile
from textSummarizer.logging import logger

In [5]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config
    
    def download_zipfile(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_url,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded! with following info: {headers}")
        else:
            logger.info(f"Filename already exists with filesize: {get_size(Path(self.config.local_data_file))}")
    
    def extract_zipfile(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

**Calling and Downloading Files**

In [6]:
try:
    config = ConfiguarationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_zipfile()
    data_ingestion.extract_zipfile()
except Exception as e:
    raise e

FileNotFoundError: [Errno 2] No such file or directory: 'config\\config.yaml'